In [12]:
import pandas as pd
import re

In [13]:
%pip install spacy


Note: you may need to restart the kernel to use updated packages.


In [14]:
import spacy
from spacy.matcher import PhraseMatcher

In [15]:
df = pd.read_csv("clean_jobs.csv")

df.head()

,job_id,job_title,company,location,job_description,experience,education,salary,job_type,clean_description,original_length,clean_length
0,JOB07402,Frontend Developer,DataBridge Analytics,"Gurgaon, India",We are looking for a Frontend Developer to joi...,1-3 years,B.E.,9-14 LPA,Full-time,we are looking for a frontend developer to joi...,362,351
1,JOB05835,Software Engineer,Vertex Digital,"Mumbai, India",We are looking for a Software Engineer to join...,3-5 years,Bachelor's Degree,12-16 LPA,Internship,we are looking for a software engineer to join...,342,333
2,JOB02123,Machine Learning Engineer,CloudSphere,"Mumbai, India",We are looking for a Machine Learning Engineer...,1-3 years,B.E.,8-11 LPA,Full-time,we are looking for a machine learning engineer...,364,354
3,JOB08789,Cloud Engineer,Apex Solutions,"Mumbai, India",We are looking for a Cloud Engineer to join ou...,3-5 years,Master's Degree,11-16 LPA,Full-time,we are looking for a cloud engineer to join ou...,361,350
4,JOB00305,Frontend Developer,Quantix Technologies,"Noida, India",We are looking for a Frontend Developer to joi...,3-5 years,B.Tech,10-12 LPA,Full-time,we are looking for a frontend developer to joi...,366,355


In [16]:
patterns = {
    "Python": r"\bpython(?:\s*3)?(?:\s+programming)?\b",
    "SQL": r"\bsql\b",
    "Java": r"\bjava\b",
    "C++": r"\bc\+\+\b",
    "JavaScript": r"\bjavascript\b",
    "Power BI": r"\bpower\s*bi\b",
    "AWS": r"\baws\b",
    "Azure": r"\bazure\b",
    "Docker": r"\bdocker\b",
    "Kubernetes": r"\bkubernetes\b"
}

In [17]:
def extract_skills_regex(text):
    found_skills = []

    text = str(text).lower()

    for skill, pattern in patterns.items():
        if re.search(pattern, text):
            found_skills.append(skill)

    return found_skills

In [18]:
df["regex_skills"] = df["clean_description"].apply(
    extract_skills_regex
)

In [19]:
test_texts = [
    "Python developer",
    "Python 3 developer",
    "Python3 programming",
    "Strong Python programming experience"
]

for text in test_texts:
    result = extract_skills_regex(text)
    print(text, "->", result)

Python developer -> ['Python']
Python 3 developer -> ['Python']
Python3 programming -> ['Python']
Strong Python programming experience -> ['Python']


In [20]:
patterns.update({
    "Machine Learning": r"\bmachine\s+learning\b",
    "Deep Learning": r"\bdeep\s+learning\b",
    "Natural Language Processing": r"\bnatural\s+language\s+processing\b",
    "Data Science": r"\bdata\s+science\b"
})

In [21]:
df["regex_skills"] = df["clean_description"].apply(
    extract_skills_regex
)

df[["job_title", "regex_skills"]].head(10)

,job_title,regex_skills
0,Frontend Developer,[JavaScript]
1,Software Engineer,"[Python, SQL]"
2,Machine Learning Engineer,"[Python, AWS, Docker, Machine Learning]"
3,Cloud Engineer,"[AWS, Azure, Docker, Kubernetes]"
4,Frontend Developer,[JavaScript]
5,Data Analyst,"[Python, SQL, Power BI]"
6,Machine Learning Engineer,"[Python, SQL, Machine Learning]"
7,DevOps Engineer,"[AWS, Kubernetes]"
8,QA Engineer,"[Python, SQL]"
9,Business Analyst,"[SQL, Power BI]"


In [22]:
nlp = spacy.blank("en")

In [23]:
matcher = PhraseMatcher(nlp.vocab, attr="LOWER")

In [24]:
skill_phrases = [
    "machine learning",
    "deep learning",
    "natural language processing",
    "power bi",
    "data science",
    "computer vision",
    "cloud computing"
]

In [25]:
phrase_patterns = [
    nlp.make_doc(skill)
    for skill in skill_phrases
]

In [26]:
matcher.add("SKILLS", phrase_patterns)

In [27]:
def extract_phrases(text):
    doc = nlp(str(text))

    matches = matcher(doc)

    found = []

    for match_id, start, end in matches:
        phrase = doc[start:end].text
        found.append(phrase)

    return found

In [28]:
text = df["clean_description"].iloc[0]

print(extract_phrases(text))

[]


In [29]:
df["phrase_skills"] = df["clean_description"].apply(
    extract_phrases
)